# 04 - Transcriptomic Analysis with CGCS

**Simulated RNA-seq Data + CGCS Scoring**  
**Allen Lab Trisomy 21 Project**

In [ ]:
# ================================================
# ROBUST SETUP - RUN THIS CELL FIRST
# ================================================
import sys
import os
from pathlib import Path

print("Current directory:", os.getcwd())

# Auto-fix directory if needed
if "allen-lab-report-tool" not in os.getcwd():
    repo_path = "/content/allen-lab-report-tool"
    if os.path.exists(repo_path):
        %cd /content/allen-lab-report-tool
    else:
        print("Cloning repository...")
        !git clone https://github.com/thinkthoughts/allen-lab-report-tool.git
        %cd allen-lab-report-tool

print("Working directory:", os.getcwd())

# Add src to path
src_path = str(Path.cwd() / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"✅ src path added: {src_path}")

# Import grok package
try:
    import grok
    from grok.trisomy_metrics import trisomy_cgcs_score
    from grok.visualization import plot_cgcs_vs_noise
    print("🎉 Grok package imported successfully!")
    print(f"Version: {grok.__version__}")
except Exception as e:
    print(f"❌ Import failed: {e}")

# Scientific libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Simulate RNA-seq / Gene Expression Data

In [ ]:
np.random.seed(42)
n_genes = 800

# Normal expression
normal_expr = np.random.normal(8.0, 1.8, n_genes)

# Trisomy 21 simulation
trisomy_expr = normal_expr.copy()
chr21_genes = np.random.choice(n_genes, 120, replace=False)   # simulated chr21 genes
trisomy_expr[chr21_genes] *= 1.5
trisomy_expr += np.random.normal(0, 0.35, n_genes)  # global noise

print(f"Simulated {n_genes} genes ({len(chr21_genes)} chr21-like genes)")

## 2. Compute CGCS from Expression Data

In [ ]:
def expression_to_cgcs(normal, trisomy):
    fc = trisomy / (normal + 1e-8)
    mean_fc = np.mean(fc)
    imbalance = np.abs(mean_fc - 1.0)
    dysregulation = np.std(fc)
    
    return trisomy_cgcs_score(
        dosage_ratio=mean_fc,
        overexpression_imbalance=imbalance,
        global_dysregulation=dysregulation,
        return_components=True
    )

result = expression_to_cgcs(normal_expr, trisomy_expr)

for key, value in result.items():
    print(f"{key:25} = {value:.4f}")

## 3. Visualizations

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(normal_expr, color='blue', alpha=0.6, kde=True, label='Normal')
sns.histplot(trisomy_expr, color='red', alpha=0.6, kde=True, label='Trisomy 21')
plt.title("Simulated Gene Expression: Normal vs Trisomy 21")
plt.xlabel("Log2 Expression")
plt.legend()
plt.show()

In [ ]:
# CGCS vs Overexpression Sweep
levels = np.linspace(1.0, 1.8, 12)
scores = []

for level in levels:
    temp = normal_expr.copy()
    temp[chr21_genes] = normal_expr[chr21_genes] * level
    score = expression_to_cgcs(normal_expr, temp)['cgcs']
    scores.append(score)

plot_cgcs_vs_noise(levels, scores, title="CGCS vs Overexpression Level (Transcriptomic)")